# Learning lab: understanding shipment risk

Start with one shipment and one question: what information could we have used to predict an incident in the next six hours?

These notebooks follow the order of the experiments, from reading the raw files to comparing models. The Python engine was built afterward. See [the plan](../plan.md) for the current status and [the runbook](../README.md) to run the finished solution.

Run this notebook from top to bottom with a Python 3.11+ kernel. These first cells use only the standard library. We inspect examples here before choosing features or fitting a model.

## 1. What does one example mean?

- **Raw event:** one delivered update, such as a temperature reading.
- **Feature:** a useful description calculated from eligible updates, such as the latest temperature.
- **Label:** the eventual answer, incident or no incident within the six hour horizon.
- **Model:** a function whose parameters are learned from examples to estimate that answer's probability.

One training row represents **one shipment at one decision time**. Many sensor updates may contribute to that row. The model does not get to see the eventual answer when making a prediction.

First, load the files without sorting or altering delivery order.


In [1]:
from pathlib import Path
from datetime import datetime, timedelta, timezone
from collections import Counter
import json

# Works when the notebook starts from this folder or the repository root.
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "data/events.jsonl").is_file() and
                (p / "src/dispatch_risk/contracts.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Open the notebook from inside the candidate repository.")

def read_jsonl(name):
    with (ROOT / "data" / name).open() as handle:
        return [json.loads(line) for line in handle if line.strip()]

def utc(text):
    value = datetime.fromisoformat(text.replace("Z", "+00:00"))
    if value.tzinfo is None:
        raise ValueError("An explicit timezone is required.")
    return value.astimezone(timezone.utc)

events = read_jsonl("events.jsonl")
labels = read_jsonl("labels.jsonl")
decisions = read_jsonl("decision_times.jsonl")
manifest = json.loads((ROOT / "data/MANIFEST.json").read_text())
print("Loaded:", len(events), "deliveries,", len(labels), "incidents,", len(decisions), "checkpoints")
print("Event kinds:", dict(Counter(e["kind"] for e in events)))
print("Distinct shipments:", len({e["shipment_id"] for e in events}))


Loaded: 11019 deliveries, 133 incidents, 1800 checkpoints
Event kinds: {'temperature_c': 10886, 'door_open': 133}
Distinct shipments: 600


## 2. Read one record from each file

An event's **device time** is the device's claimed measurement time. Its **received time** is when the platform could first use that revision. These are different facts.

Labels also have two times: when the incident happened and when its audited report became available. A late report can establish a historical outcome, but it cannot have been used to fit a model before the report existed.


In [2]:
for name, records in [("EVENT", events), ("DECISION CHECKPOINT", decisions), ("INCIDENT", labels)]:
    print("\n" + name)
    print(json.dumps(records[0], indent=2) if records else "No records")



EVENT
{
  "device_time": "2026-01-01T03:00:00Z",
  "event_id": "s-00000-temp-03",
  "kind": "temperature_c",
  "payload": {
    "firmware": "4.7.9"
  },
  "received_at": "2026-01-01T03:35:00Z",
  "revision": 1,
  "shipment_id": "s-00000",
  "source": "sensor-north",
  "value": 3.737
}

DECISION CHECKPOINT
{
  "decision_time": "2026-01-01T08:00:00Z",
  "shipment_id": "s-00000"
}

INCIDENT
{
  "incident_at": "2026-01-01T17:00:00Z",
  "incident_id": "inc-00000",
  "label_available_at": "2026-01-02T11:00:00Z",
  "severity": 1,
  "shipment_id": "s-00000"
}


## 3. Why can't we treat every line as a new measurement?

A record can be delivered twice. A correction is a different revision of an existing event. We count these separately to understand the dataset. This cell is an audit, not the engine's deduplication implementation.


In [3]:
identities = Counter((e["event_id"], e["revision"]) for e in events)
versions = {}
for event in events:
    versions.setdefault(event["event_id"], set()).add(event["revision"])
print("Repeated identity deliveries:", sum(count - 1 for count in identities.values()))
print("Event IDs with multiple revisions:", sum(len(v) > 1 for v in versions.values()))
print("Adjacent delivery pairs with decreasing received time:",
      sum(utc(b["received_at"]) < utc(a["received_at"])
          for a, b in zip(events, events[1:])))
print("A repeated identity is not automatically proof that its payload is identical.")


Repeated identity deliveries: 1200
Event IDs with multiple revisions: 86
Adjacent delivery pairs with decreasing received time: 441
A repeated identity is not automatically proof that its payload is identical.


## 4. Follow one real shipment

Choose the shipment at the earliest checkpoint in the supplied data; do not hard-code its ID. The table retains original file delivery order. We show only that shipment's rows.

**Questions:** Does measurement order match delivery order? Which records are redeliveries? Which are corrections? Does any operational update claim an old device time but arrive much later?


In [4]:
first_decision = min(decisions, key=lambda d: (utc(d["decision_time"]), d["shipment_id"]))
shipment_id = first_decision["shipment_id"]
shipment_events = [(i, e) for i, e in enumerate(events) if e["shipment_id"] == shipment_id]
shipment_decisions = sorted((d for d in decisions if d["shipment_id"] == shipment_id),
                            key=lambda d: utc(d["decision_time"]))
print("Shipment:", shipment_id)
print("Checkpoints:", [d["decision_time"] for d in shipment_decisions])
print("file index | event | rev | device time | received time | kind | value")
for i, e in shipment_events:
    print(i, e["event_id"], e["revision"], e["device_time"], e["received_at"],
          e["kind"], e["value"], sep=" | ")


Shipment: s-00000
Checkpoints: ['2026-01-01T08:00:00Z', '2026-01-01T11:00:00Z', '2026-01-01T14:00:00Z']
file index | event | rev | device time | received time | kind | value
0 | s-00000-temp-03 | 1 | 2026-01-01T03:00:00Z | 2026-01-01T03:35:00Z | temperature_c | 3.737
1 | s-00000-temp-01 | 1 | 2026-01-01T01:00:00Z | 2026-01-01T01:01:00Z | temperature_c | 3.505
2 | s-00000-temp-02 | 1 | 2026-01-01T02:00:00Z | 2026-01-01T02:02:00Z | temperature_c | 3.786
3 | s-00000-temp-00 | 1 | 2026-01-01T00:00:00Z | 2026-01-01T00:05:00Z | temperature_c | 4.224
5 | s-00000-temp-04 | 1 | 2026-01-01T04:00:00Z | 2026-01-01T04:01:00Z | temperature_c | 3.399
6 | s-00000-temp-04 | 1 | 2026-01-01T04:00:00Z | 2026-01-01T04:01:00Z | temperature_c | 3.399
8 | s-00000-temp-05 | 1 | 2026-01-01T05:00:00Z | 2026-01-01T05:01:00Z | temperature_c | 5.038
10 | s-00000-temp-06 | 1 | 2026-01-01T06:00:00Z | 2026-01-01T06:00:00Z | temperature_c | 4.687
15 | s-00000-temp-07 | 1 | 2026-01-01T07:00:00Z | 2026-01-01T07:35:00Z | 

## 5. Freeze what was available at one checkpoint

This first experiment separates **available** and **not yet available** deliveries using received time. It is only the first eligibility gate: duplicate resolution, revision selection, bad-clock handling, and past period policies come later.

For historical dataset reconstruction, availability is defined by the supplied received timestamp. Online replay additionally has access only to deliveries already ingested. Those two contexts must not be conflated.


In [5]:
as_of = utc(first_decision["decision_time"])
available = [e for _, e in shipment_events if utc(e["received_at"]) <= as_of]
not_yet_available = [e for _, e in shipment_events if utc(e["received_at"]) > as_of]
print("Decision time:", as_of.isoformat())
print("Available deliveries (not yet deduplicated):", len(available))
print("Not yet available:", len(not_yet_available))
for e in not_yet_available:
    if utc(e["device_time"]) <= as_of:
        print("Looks old but was not known:", e["event_id"], "revision", e["revision"],
              "received", e["received_at"])
assert all(utc(e["received_at"]) <= as_of for e in available)


Decision time: 2026-01-01T08:00:00+00:00
Available deliveries (not yet deduplicated): 9
Not yet available: 11
Looks old but was not known: s-00000-temp-08 revision 1 received 2026-01-01T11:00:00Z
Looks old but was not known: s-00000-temp-08 revision 2 received 2026-01-01T19:00:00Z


## 6. Locate eventual outcomes, without leaking them into features

The horizon is open on the left and closed on the right: an incident exactly at the checkpoint is excluded; one exactly six hours later is included.

This cell inspects recorded incidents. It deliberately does **not** turn absence into a verified negative label. The files do not supply an explicit end-of-observation field or a guaranteed maximum reporting delay. We must establish a defensible completeness/censoring policy before producing general training labels.


In [6]:
for decision in shipment_decisions:
    start = utc(decision["decision_time"])
    end = start + timedelta(hours=6)
    matches = [label for label in labels if label["shipment_id"] == shipment_id
               and start < utc(label["incident_at"]) <= end]
    print("\nCheckpoint:", start.isoformat(), "through", end.isoformat())
    if matches:
        for label in matches:
            print("Recorded positive:", label["incident_at"],
                  "report available:", label["label_available_at"])
    else:
        print("No incident listed in this window; negative eligibility needs a completeness policy.")



Checkpoint: 2026-01-01T08:00:00+00:00 through 2026-01-01T14:00:00+00:00
No incident listed in this window; negative eligibility needs a completeness policy.

Checkpoint: 2026-01-01T11:00:00+00:00 through 2026-01-01T17:00:00+00:00
Recorded positive: 2026-01-01T17:00:00Z report available: 2026-01-02T11:00:00Z

Checkpoint: 2026-01-01T14:00:00+00:00 through 2026-01-01T20:00:00+00:00
Recorded positive: 2026-01-01T17:00:00Z report available: 2026-01-02T11:00:00Z


## 7. Propose features before calculating them

| Candidate | Question it helps answer | Policy still needed |
|---|---|---|
| Latest temperature | What was the latest observed condition? | Eligible revision and usable measurement time |
| Trailing mean/max | Was it consistently warm or was there a spike? | Lookback window and sparse data |
| Temperature trend | Is it warming or cooling? | Irregular timing and minimum observations |
| Measurement age | How stale is our information? | Bad/future device clocks |
| Reading count | How much evidence do we have? | Deduplication and revisions |
| Missing-temperature flag | Do we lack usable temperature data? | Missing values versus actual zero |

A hypothesis is not an established relationship. We will test whether these features help. We will not invent a universal temperature threshold or fit preparing model inputs using the evaluation data.

### Check your understanding
1. Why is one event not the same thing as one training example?
2. Why can't an old device timestamp make a late delivery usable earlier?
3. How is a correction different from a duplicate?
4. Why doesn't a missing incident record automatically prove a negative outcome?

### Next session
Resolve duplicates and revisions for one historical checkpoint; decide how clocks and feature windows should behave. Then calculate a few features by hand and in code. Model training begins after the feature and label policies are understood.

**Later lessons:** logistic regression and tree alternatives; training/loss/regularization; ordered by time evaluation and probability metrics; artifact export; production engine; tests of required behavior and interview practice. Refer to `../plan.md` for completion criteria.


## Next checkpoint

Continue in [02_duplicates_and_revisions.ipynb](02_duplicates_and_revisions.ipynb). Revision experiments have moved there. Each checkpoint now has its own independently runnable notebook.
